# 04 · Do the synergies predict hand pose? (Q3)

**Owner:** Person D · **Read first:** Lecture 16 p. 13–15 (linear decoders), reading 1027 (Paninski),
[Hadidi et al. 2026](https://arxiv.org/abs/2603.08212) (position vs velocity decoding on this dataset)

**We expect:** pose from the k synergy activations is nearly as good as from all 16 channels, and clearly
better than "always guess the average pose". Use only frames where `pose_valid` is True.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Goal:** answer **Q3, are the synergies useful?** Test whether they carry the information needed to predict hand pose.

**Why:** synergies that are real but can't predict anything are a curiosity. If k synergies predict pose
nearly as well as all 16 channels, they're a genuine compressed description of how the hand is controlled.

**Milestones**
1. Baseline: always predict the average pose (the number to beat)
2. Ridge regression from all 16 channels (the ceiling)
3. Ridge regression from the k synergy activations c(t)
4. Compare all three with angular error (degrees) and R²

**Done when:** one bar chart of baseline vs 16 channels vs synergies.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01 (data/ is never committed)

In [ ]:
# ── Load the shared data made by notebook 01 (runs 01 first if needed) ───────
import numpy as np
if not PROCESSED.exists():
    %run notebooks/experiments/01_data_prep/01_data_prep.ipynb
d = np.load(PROCESSED)

emg        = d["emg"]          # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"]         # (N, 20)  joint angles in degrees
pose_valid = d["pose_valid"]   # (N,)     False where pose tracking failed
rec        = d["rec"]          # (N,)     which recording (0–29) each row comes from
split      = d["split"]        # (N,)     0 = train, 1 = test, -1 = gap (use neither)
stage_of_rec, side_of_rec = d["stage"], d["side"]   # (30,) task name and hand per recording

train, test = split == 0, split == 1
SEED = 0   # pass random_state=SEED to every model so results repeat
print(f"{len(emg)} frames | {train.sum()} train | {test.sum()} test | {len(stage_of_rec)} recordings")

K = 4   # TODO: replace with K_CHOSEN from notebook 02
tr, te = train & pose_valid, test & pose_valid

### The variables you start with
`N` = number of frames in all 30 recordings together (≈ 90,000: 30 recordings × ~60 s × 50 frames/s).
Every row of every array is one frame, 20 ms long. Boolean arrays are **masks**: `emg[train]` keeps only the train rows.

| Name | Shape | dtype | What it is |
|---|---|---|---|
| `emg` | (N, 16) | float32 | EMG envelope, one column per electrode, ≥ 0, roughly 0–1 |
| `pose` | (N, 20) | float32 | joint angles in **degrees**, one column per joint |
| `pose_valid` | (N,) | bool | False where pose tracking failed (don't use those `pose` rows) |
| `rec` | (N,) | int16 | which recording (0–29) the frame comes from |
| `split` | (N,) | int8 | 0 = train, 1 = test, −1 = gap between them (use neither) |
| `train`, `test` | (N,) | bool | `split == 0`, `split == 1` |
| `stage_of_rec` | (30,) | str | task name of each recording; per frame: `stage_of_rec[rec]` → (N,) |
| `side_of_rec` | (30,) | str | `"left"` or `"right"` per recording; per frame: `side_of_rec[rec]` → (N,) |

Letters used in the docstrings below: **T** = number of frames in the array you pass in (any subset of N),
**k** = number of synergies, **C** = number of gesture classes. Run a helper on a tiny made-up array first
(e.g. `np.random.rand(5, 16)`) to check it returns the shape its docstring promises.

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

## Task 1 · Metrics
- **Angular error (°):** mean absolute difference in joint angle (the unit published papers use)
- **R²:** fraction of pose variation explained (`sklearn.metrics.r2_score`)

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

def angular_error(Y: np.ndarray, Y_hat: np.ndarray) -> float:
    """Mean absolute joint-angle error.

    Parameters
    ----------
    Y     : (T, 20) float, true joint angles in degrees
    Y_hat : (T, 20) float, predicted joint angles in degrees

    Returns
    -------
    float, degrees, averaged over all joints and frames. Check: angular_error(Y, Y) → 0.0
    """
    # TODO
    raise NotImplementedError


def score(Y: np.ndarray, Y_hat: np.ndarray) -> dict[str, float]:
    """Both metrics at once. Y, Y_hat: (T, 20) degrees.

    Returns {"angle_deg": angular_error(...), "r2": r2_score(Y, Y_hat)}.
    """
    # TODO
    raise NotImplementedError


tr = train & pose_valid     # (N,) bool: training frames with a usable pose
te = test  & pose_valid     # (N,) bool: test frames with a usable pose
results: dict[str, dict[str, float]] = {}   # e.g. results["baseline"] = {"angle_deg": 12.3, "r2": 0.0}

## Task 2 · Baseline: always predict the median training pose

In [ ]:
median_pose = None   # (20,) float: the median of pose[tr] over frames, one value per joint
Y_hat_base  = None   # (T_test, 20): median_pose repeated for every test frame (np.tile)
# TODO: fill both, then results["baseline"] = score(pose[te], Y_hat_base)

## Task 3 · Ridge regression from all 16 channels (the simple-model ceiling)

In [ ]:
from sklearn.linear_model import Ridge

def fit_and_score_ridge(F_train: np.ndarray, Y_train: np.ndarray,
                        F_test: np.ndarray, Y_test: np.ndarray, alpha: float = 1.0) -> dict[str, float]:
    """Linear decoder: features → pose. Fit on train, score on test.

    Parameters
    ----------
    F_train : (T_train, d) float, input features (d = 16 for all channels, k for synergies)
    Y_train : (T_train, 20) float, joint angles in degrees
    F_test  : (T_test, d) float
    Y_test  : (T_test, 20) float
    alpha   : ridge penalty (try 0.1, 1, 10 on the TRAIN data only, never by peeking at test)

    Returns
    -------
    score(Y_test, prediction): {"angle_deg": …, "r2": …}
    """
    # TODO
    raise NotImplementedError


# TODO: results["all 16 channels"] = fit_and_score_ridge(emg[tr], pose[tr], emg[te], pose[te])

## Task 4 · Ridge regression from the k synergy activations c(t)
Fit NMF on `emg[train]` only, then c = `nmf.transform(...)` for train and test.

In [ ]:
from sklearn.decomposition import NMF

K = 4   # replace with K_CHOSEN from notebook 02

nmf = None       # TODO: NMF(n_components=K, random_state=SEED, max_iter=1000) fitted on emg[train]  (all train frames)
c_tr = None      # (T_train_valid, K): nmf.transform(emg[tr])
c_te = None      # (T_test_valid, K):  nmf.transform(emg[te])
# TODO: results[f"{K} synergies"] = fit_and_score_ridge(c_tr, pose[tr], c_te, pose[te])

## Task 5 · Compare
**Figure:** one bar chart of angular error (and one of R²) for baseline, 16 channels, synergies.
*Extra if time:* the same with k PCA components instead of synergies.

In [ ]:
# results now has 3 entries: name → {"angle_deg": float, "r2": float}
# TODO: two bar charts side by side: angle_deg per entry, and r2 per entry
# TODO (extra): the same as Task 4 with PCA(n_components=K), added as results[f"{K} PCs"]

## Result
| Input | Angular error (°) | R² |
|---|---|---|
| Baseline (median pose) | | |
| All 16 channels | | |
| k synergies | | |